In [ ]:
from pathlib import Path
import sys
root = Path.cwd().resolve()
while not (root / "src" / "config.py").exists():
    if root == root.parent:
        raise RuntimeError("Open this notebook within the project directory")
    root = root.parent
sys.path.insert(0, str(root))
from src.config import Config
from src.utils import device
print(device())
config = Config()


# Held-out FLAIR threshold baseline

In [ ]:
from src.baseline import run_baseline
run_baseline(config)

## FLAIR threshold (tuned on development patients)

This separate classical comparator uses raw FLAIR channel 1, a filled largest-component brain mask, Euclidean erosion, patient-pooled intensity statistics, slice component removal, and optional largest 26-connected volume component. The fixed 440-setting grid is selected using mean patient-volume Dice on the 74 training + 14 validation patients only. Exact ties favor smaller erosion, smaller minimum area, no 3D filtering, then smaller k.

The config is saved before opening test files. Completed patient measurements are cached, and completed runs reuse results rather than reevaluating test patients. File lists and numeric order are asserted against the archived U-Net evaluation. This comparison describes one split of 22 previously inspected patients and does not establish a general U-Net advantage. The original spec-baseline cell and artifacts remain separate.

Run only the new cells to inspect the tuned comparator without rerunning the original baseline. CLI: `python -m src.baseline_tuned`. Isolated synthetic smoke: `python scripts/smoke_test_baseline_tuned.py --synthetic`.


In [ ]:
import json
import pandas as pd
from IPython.display import display, Image
from src.baseline_tuned import run_tuned_baseline

tuned_scores = run_tuned_baseline(config)
tuned_metrics = json.loads((config.output_dir / "metrics.json").read_text())["baseline_tuned"]
print("Frozen parameters:", tuned_metrics["parameters"])
display(pd.DataFrame({name: tuned_metrics[name] for name in ["development", "test"]}))
display(tuned_scores[["patient", "baseline_tuned_dice", "unet_dice"]])
for patient in ["DU_6407", "DU_7018"]:
    display(Image(filename=str(config.output_dir / f"baseline_tuned_{patient}.png")))
